# Schema parquet `churn_customers` (1 part)

Đọc kiểu **trong file** (PyArrow) và kiểu **sau khi load pandas**.

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

PARQUET_PATH = Path("../churn_customers/part-00000-890427e8-5a4c-4000-9761-912870770d03-c000.snappy.parquet")
if not PARQUET_PATH.exists():
    PARQUET_PATH = Path("churn_customers/part-00000-890427e8-5a4c-4000-9761-912870770d03-c000.snappy.parquet")

print(PARQUET_PATH.resolve())
print("exists", PARQUET_PATH.exists())

C:\Homework\AI-Intern\temp\churn_customers\part-00000-890427e8-5a4c-4000-9761-912870770d03-c000.snappy.parquet
exists True


## 1. Kiểu lưu trong parquet (PyArrow) — nguồn sự thật

In [2]:
pf = pq.ParquetFile(PARQUET_PATH)
schema = pf.schema_arrow
print(schema)
print("\nfield | arrow type | logical type")
for field in schema:
    print(f"{field.name:20} {str(field.type):20} {field.type}")

customer_id: int32
gender: string
birth_date: date32[day]
region: string
city: string
signup_date: date32[day]
account_status: string
closed_date: date32[day]
last_login_at: timestamp[ns]
-- schema metadata --
org.apache.spark.version: '3.5.6'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 605

field | arrow type | logical type
customer_id          int32                int32
gender               string               string
birth_date           date32[day]          date32[day]
region               string               string
city                 string               string
signup_date          date32[day]          date32[day]
account_status       string               string
closed_date          date32[day]          date32[day]
last_login_at        timestamp[ns]        timestamp[ns]


## 2. Kiểu sau `pd.read_parquet` (pandas có thể đổi DATE → object/`datetime.date`)

In [4]:
df = pd.read_parquet(PARQUET_PATH)
print("shape", df.shape)
print("\npandas dtypes")
print(df.dtypes)
print("\nPython type của ô đầu mỗi cột")
for col in df.columns:
    val = df[col].iloc[0]
    print(f"{col:20} {type(val).__module__}.{type(val).__name__}  sample={val!r}")
df.head(3)

shape (10002, 9)

pandas dtypes
customer_id                int32
gender                    object
birth_date                object
region                    object
city                      object
signup_date               object
account_status            object
closed_date               object
last_login_at     datetime64[ns]
dtype: object

Python type của ô đầu mỗi cột
customer_id          numpy.int32  sample=np.int32(1)
gender               builtins.str  sample='M'
birth_date           datetime.date  sample=datetime.date(1991, 5, 19)
region               builtins.str  sample='Mien Nam'
city                 builtins.str  sample='Ho Chi Minh'
signup_date          datetime.date  sample=datetime.date(2025, 3, 24)
account_status       builtins.str  sample='Active'
closed_date          builtins.NoneType  sample=None
last_login_at        pandas._libs.tslibs.timestamps.Timestamp  sample=Timestamp('2026-07-26 09:27:00')


,customer_id,gender,birth_date,region,city,signup_date,account_status,closed_date,last_login_at
0,1,M,1991-05-19,Mien Nam,Ho Chi Minh,2025-03-24,Active,None,2026-07-26 09:27:00
1,2,M,1979-12-31,Mien Bac,Quang Ninh,2025-05-15,Active,None,2025-11-21 03:19:00
2,3,M,1987-07-07,Mien Nam,Can Tho,2025-12-16,Active,None,2026-07-02 17:00:00
